In [1]:
import sys
from pathlib import Path

import numpy as np
import open3d as o3d

ROOT_DIR = next(
    directory
    for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / "src").is_dir() and (directory / "data").is_dir()
)
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

PROCESSED_DIR = ROOT_DIR / "data/processed"
REPORT_IMAGES_DIR = ROOT_DIR / "report/report_images"

from src.utils.plotting import show_geometries, save_open3d_screenshot

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Paso 0 - cargar nube global

In [2]:
ply_path = PROCESSED_DIR / "palacio" / "nube_global_palacio.ply"
pcd = o3d.io.read_point_cloud(str(ply_path))

initial_point_count = len(pcd.points)
print(f"Puntos iniciales: {initial_point_count:,}")

# Guardamos cuántos puntos quedan en cada paso para comparar después.
point_counts = {"inicial": initial_point_count}

show_geometries([pcd])

Puntos iniciales: 17,703,320


## Paso 1 - Filtrado espacial (recorte de caja)

In [3]:
points = np.asarray(pcd.points)
print("min (x,y,z):", points.min(axis=0))
print("max (x,y,z):", points.max(axis=0))

min (x,y,z): [-2909.03190151 -3111.85087307 -1250.57029688]
max (x,y,z): [3171.49749619 3842.34727798 2604.7993085 ]


In [4]:
min_bound = np.array([130.0, 90.0, -140.0])  
max_bound = np.array([ 220.0,  470.0,  0.0])    
bbox = o3d.geometry.AxisAlignedBoundingBox(min_bound, max_bound)
bbox.color = (1, 0, 0)

pcd_bbox = pcd.crop(bbox)

point_counts["bbox"] = len(pcd_bbox.points)
print(f"Antes:   {len(pcd.points):,}")
print(f"Después: {point_counts['bbox']:,}")

show_geometries([pcd_bbox])

Antes:   17,703,320
Después: 225,228


## Paso 2 - Submuestreo (voxel downsample)

In [5]:
voxel_size = 1.5   # mm — más chico conserva más detalle y sube el peso del .ply
pcd_voxel = pcd_bbox.voxel_down_sample(voxel_size=voxel_size)

point_counts["voxel"] = len(pcd_voxel.points)
print(f"Tras voxel ({voxel_size} mm): {point_counts['voxel']:,} puntos")
show_geometries([pcd_voxel])

Tras voxel (1.5 mm): 214,977 puntos


## Paso 3 - Eliminación de outliers

In [6]:
clean_point_cloud, idx = pcd_voxel.remove_statistical_outlier(
    nb_neighbors=20,
    std_ratio=1.5,
)

point_counts["outliers"] = len(clean_point_cloud.points)
print(f"Tras outliers: {point_counts['outliers']:,} puntos")
show_geometries([clean_point_cloud])

Tras outliers: 202,741 puntos


In [7]:
# Para el informe, pintamos en rojo lo que el filtro dejó afuera.
outliers = pcd_voxel.select_by_index(idx, invert=True)   
outliers.paint_uniform_color([1, 0, 0])
show_geometries([clean_point_cloud, outliers])

### Puntos sobrevivientes por etapa

In [8]:
# Tabla de puntos por etapa (para el informe)
print(f"{'etapa':>10} | {'puntos':>12} | {'% del inicial':>13}")
print("-" * 42)
for stage, n in point_counts.items():
    print(f"{stage:>10} | {n:>12,} | {100 * n / point_counts['inicial']:>12.3f}%")

     etapa |       puntos | % del inicial
------------------------------------------
   inicial |   17,703,320 |      100.000%
      bbox |      225,228 |        1.272%
     voxel |      214,977 |        1.214%
  outliers |      202,741 |        1.145%


In [9]:
# Capturas de cada etapa para el informe
REPORT_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

for point_cloud, name in [
    (pcd_bbox, "palacio_bbox"),
    (pcd_voxel, "palacio_voxel"),
    (clean_point_cloud, "palacio_optimizada"),
]:
    save_open3d_screenshot([point_cloud], str(REPORT_IMAGES_DIR / f"{name}.png"))
    print("Guardada:", REPORT_IMAGES_DIR / f"{name}.png")

Guardada: /Users/mathiaskneipp/Documents/Developer/UdeSA/Cuatrimestre_6/Vision_Artificial/vision-tp2/report/report_images/palacio_bbox.png
Guardada: /Users/mathiaskneipp/Documents/Developer/UdeSA/Cuatrimestre_6/Vision_Artificial/vision-tp2/report/report_images/palacio_voxel.png
Guardada: /Users/mathiaskneipp/Documents/Developer/UdeSA/Cuatrimestre_6/Vision_Artificial/vision-tp2/report/report_images/palacio_optimizada.png


## Paso 4 - Exportar el .ply

In [10]:
output_path = PROCESSED_DIR / "palacio" / "nube_palacio_optimizada.ply"
o3d.io.write_point_cloud(str(output_path), clean_point_cloud, write_ascii=False)   # binario
print("Guardado en:", output_path)
print("Tamaño:", round(output_path.stat().st_size / 1e6, 2), "MB")

Guardado en: /Users/mathiaskneipp/Documents/Developer/UdeSA/Cuatrimestre_6/Vision_Artificial/vision-tp2/data/processed/palacio/nube_palacio_optimizada.ply
Tamaño: 5.47 MB


## Paso 5 - Medición métrica

In [11]:
aabb = clean_point_cloud.get_axis_aligned_bounding_box()
dx, dy, dz = np.round(aabb.get_extent(), 1)
print("== AABB (ejes del mundo) ==")
print(f"  ancho  (x): {dx:7.1f} mm")
print(f"  largo  (y): {dy:7.1f} mm")
print(f"  alto   (z): {dz:7.1f} mm")

# La AABB sigue los ejes del mundo; esta caja acompaña la orientación del objeto.
obb = clean_point_cloud.get_minimal_oriented_bounding_box()
ex, ey, ez = np.round(np.sort(obb.extent)[::-1], 1)  # De mayor a menor, porque la caja puede quedar rotada.
print("\n== OBB (ajustada al objeto) ==")
print(f"  dimensiones (mayor->menor): {ex:.1f} x {ey:.1f} x {ez:.1f} mm")

actual_height_mm = None   # p. ej. 135.0
if actual_height_mm is not None:
    relative_error = 100 * abs(dz - actual_height_mm) / actual_height_mm
    print(f"\nAlto estimado: {dz:.1f} mm | real: {actual_height_mm:.1f} mm "
          f"| error relativo: {relative_error:.1f}%")

# Visualización de la medición
obb.color = (1, 0, 0)
show_geometries([clean_point_cloud, obb])

== AABB (ejes del mundo) ==
  ancho  (x):    90.0 mm
  largo  (y):   380.0 mm
  alto   (z):   129.8 mm

== OBB (ajustada al objeto) ==
  dimensiones (mayor->menor): 380.0 x 129.8 x 90.0 mm
